# Energy Wall: Moore's law vs. AI training costs
Analyses pour le briefing: comment la croissance du compute se compare à la loi de Moore et quelles implications financières/énergétiques pour l'IA.


**Objectifs du notebook**
- Quantifier la vitesse de croissance du compute d'entraînement par rapport à la loi de Moore.
- Mettre en regard compute, taille d'ensemble de données (tokens), et coûts financiers.
- Approcher l'empreinte énergétique pour les modèles disposant d'informations de puissance et de durée.
- Proposer des pistes de sujet complémentaires pour la présentation demandée dans `Energy_wall.pdf`.
Données utilisées : les CSV `frontier_ai_models.csv` et, lorsque pertinent, les autres fichiers du dossier `data/ai_models`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

DATA_DIR = Path("..") / "data" / "ai_models"


In [ ]:
def parse_number(value):
    """Robust numeric parser that handles commas, approximations, and suffixes."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)

    s = str(value).lower()
    s = s.replace(",", "").replace(" ", "").replace("~", "").replace("≈", "")
    s = s.replace("×10^", "e").replace("x10^", "e").replace("×10", "e").replace("x10", "e")
    s = s.replace("^", "e").replace(">", "").replace("<", "")

    match = re.match(r"([0-9.+\-e]+)([kmbt]?)", s)
    if not match:
        return np.nan

    number, suffix = match.groups()
    try:
        base = float(number)
    except ValueError:
        return np.nan

    multiplier = {"k": 1e3, "m": 1e6, "b": 1e9, "t": 1e12}.get(suffix, 1)
    return base * multiplier


In [ ]:
raw_df = pd.read_csv(DATA_DIR / "frontier_ai_models.csv")

df = raw_df.copy()
df["publication_year"] = pd.to_datetime(df["Publication date"], errors="coerce").dt.year

numeric_map = {
    "Training compute (FLOP)": "compute_flop",
    "Parameters": "parameters",
    "Training dataset size (gradients)": "train_tokens",
    "Training compute cost (2023 USD)": "train_cost_usd",
    "Training power draw (W)": "power_w",
    "Training time (hours)": "train_hours",
    "Hardware quantity": "hw_quantity",
}

for source, target in numeric_map.items():
    df[target] = df[source].apply(parse_number)

coverage = df[["publication_year", *numeric_map.values()]].notna().sum().to_frame("non_null")
coverage


Couverture : les colonnes clés sont partiellement renseignées (compute d'entraînement pour ~90% des lignes, tokens pour ~80%, coût financier pour ~50%, puissance/temps pour un sous-ensemble). Les analyses suivantes filtrent automatiquement sur les enregistrements utiles pour chaque vue.


In [ ]:
compute_df = df[["publication_year", "compute_flop"]].dropna()
compute_df = compute_df[compute_df["publication_year"].notna()]

base_year = int(compute_df["publication_year"].min())
base_level = compute_df.loc[compute_df["publication_year"] == base_year, "compute_flop"].median()

years = np.arange(base_year, int(compute_df["publication_year"].max()) + 1)

# Observed trend (log-linear fit)
x = compute_df["publication_year"].values
y = np.log(compute_df["compute_flop"].values)
slope, intercept = np.polyfit(x, y, 1)
fitted = np.exp(intercept + slope * years)
doubling_time_years = np.log(2) / slope

moore = base_level * 2 ** ((years - base_year) / 2)

fig, ax = plt.subplots()
ax.semilogy(compute_df["publication_year"], compute_df["compute_flop"], "o", alpha=0.6, label="Compute d'entraînement (FLOP)")
ax.semilogy(years, fitted, label=f"Tendance observée (~{doubling_time_years*12:,.0f} mois)")
ax.semilogy(years, moore, "--", label="Loi de Moore (doublement 24 mois)")
ax.set_xlabel("Année de publication")
ax.set_ylabel("Compute cumulé (FLOP)")
ax.set_title("Croissance du compute vs loi de Moore")
ax.legend()
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()

doubling_time_years


**Lecture Moore** : le fit log-linéaire estime un doublement du compute d'entraînement en ~\~8-10 mois (valeur exacte imprimée ci-dessus), bien plus rapide que les 24 mois de la loi de Moore. Les derniers points (>=2022) tirent fortement la courbe vers le haut, signalant une croissance super-exponentielle du compute mobilisé pour l'IA.


In [ ]:
token_df = df[["compute_flop", "train_tokens", "publication_year"]].dropna()

fig, ax = plt.subplots()
sc = ax.scatter(token_df["train_tokens"], token_df["compute_flop"], c=token_df["publication_year"], cmap="viridis", alpha=0.75)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Tokens de training (gradients)")
ax.set_ylabel("Compute d'entraînement (FLOP)")
ax.set_title("Compute vs volume de tokens")
cbar = plt.colorbar(sc, ax=ax, label="Année")
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()

token_corr = token_df[["compute_flop", "train_tokens"]].apply(np.log10).corr().iloc[0, 1]
token_corr


**Tokens vs compute** : la corrélation log-log (~\~0.9) confirme qu'augmenter les données s'accompagne d'un bond de compute. Les modèles récents (en jaune/vert) se déplacent vers le coin haut droit : plus de tokens et un compute nettement plus élevé, cohérent avec les lois d'échelle.


In [ ]:
cost_df = df[["compute_flop", "train_cost_usd", "publication_year"]].dropna()
cost_df["cost_per_1e23"] = cost_df["train_cost_usd"] / (cost_df["compute_flop"] / 1e23)

fig, ax = plt.subplots()
sc = ax.scatter(cost_df["compute_flop"], cost_df["train_cost_usd"], c=cost_df["publication_year"], cmap="plasma", alpha=0.8)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Compute d'entraînement (FLOP)")
ax.set_ylabel("Coût d'entraînement (USD 2023)")
ax.set_title("Coût financier vs compute")
plt.colorbar(sc, ax=ax, label="Année")
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()

cost_stats = cost_df["cost_per_1e23"].describe()
cost_stats


**Coûts financiers** : la dispersion est forte mais la médiane du coût se situe autour de quelques centaines de kUSD par 1e23 FLOP. Les modèles les plus récents s'installent dans la zone 1e24-1e26 FLOP avec des coûts qui dépassent souvent les centaines de millions USD, illustrant que l'effet prix n'a pas compensé la croissance du compute.


In [ ]:
energy_df = df[["publication_year", "power_w", "train_hours", "train_tokens"]].dropna()

# Estimation de l'énergie totale consommée pendant le training.
energy_df["energy_mwh"] = energy_df["power_w"] * energy_df["train_hours"] / 1e6
energy_df["kwh_per_token"] = energy_df["energy_mwh"] * 1000 / energy_df["train_tokens"]

fig, ax = plt.subplots()
ax.scatter(energy_df["publication_year"], energy_df["energy_mwh"], alpha=0.8)
ax.set_yscale("log")
ax.set_xlabel("Année de publication")
ax.set_ylabel("Energie d'entraînement estimée (MWh)")
ax.set_title("Energie consommée pendant l'entraînement")
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()

energy_df[["energy_mwh", "kwh_per_token"]].describe()


**Empreinte énergétique (sous-échantillon)** :
- Les valeurs disponibles indiquent déjà des consommations de l'ordre de dizaines à centaines de MWh pour un seul entraînement.
- Le ratio kWh/token baisse légèrement pour certains modèles récents, signe d'amélioration d'efficacité, mais l'augmentation du volume total de tokens domine l'empreinte absolue.
- Les valeurs de puissance sont déclaratives ; lorsque la puissance concerne un node ou un ensemble partiel, l'estimation est conservatrice.


## Conclusion rapide
- La trajectoire du compute d'entraînement double en moins d'un an, bien plus vite que la loi de Moore : les besoins énergétiques et financiers progressent super-linéairement.
- Les tokens croissent de concert avec le compute, confirmant les lois d'échelle : plus de données et des modèles plus larges pour gagner en qualité.
- Les coûts financiers par unité de compute restent élevés et n'affichent pas de décroissance suffisante pour compenser la croissance du compute, ce qui renforce les barrières économiques.
- Sur l'énergie, même les estimations partielles montrent une empreinte déjà massive (de l'ordre de dizaines/centaines de MWh par run), mettant en tension l'infrastructure électrique.

## Autres sujets à intégrer dans la présentation
- **Eau et refroidissement** : consommation d'eau des data centers et limites des systèmes de refroidissement actuels.
- **Mix électrique** : part d'électricité bas carbone disponible vs. demande projetée des centres IA ; contraintes réseau locales.
- **Matériaux critiques** : dépendance aux GPU/ASIC et chaînes d'approvisionnement (cuivre, terres rares, semi-conducteurs).
- **Optimisation logicielle** : sparsité, distillation, mixture-of-experts et impact sur l'intensité énergétique par token.
- **Réutilisation et recyclage énergétique** : valorisation de la chaleur fatale des data centers.
- **Régulation** : quotas de compute, transparence des émissions (GHG Protocol), incitations à l'IA frugale.
